# 🗂️ Unidad 2 — Estructuras de Datos
## Semana 4 · Unidad 2 — Tipos de Datos Abstractos y Union-Find

---

> **Curso:** Algoritmos y Estructuras de Datos · PhD. César Astudillo  
> **Duración:** 2 horas  
> **Prerequisitos:** Fundamentos de Python, listas, funciones

---

### 🗺️ Mapa de la clase

| # | Tema | Tiempo estimado |
|---|------|-----------------|
| 1 | ¿Qué es un Tipo de Dato Abstracto (TDA)? | 15 min |
| 2 | El problema de conectividad dinámica | 10 min |
| 3 | API de Union-Find | 10 min |
| 4 | Quick Find | 25 min |
| 5 | Quick Union | 25 min |
| 6 | Dos mejoras: unión por tamaño y compresión de caminos | 10 min |
| 7 | Comparación y análisis de costo | 10 min |
| 8 | Ejercicios | 15 min |

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Comprender** qué es un Tipo de Dato Abstracto y por qué separa la interfaz de la implementación.
2. **Identificar** el problema de conectividad dinámica y modelarlo con las operaciones `union` y `find`.
3. **Implementar** Quick-Find y Quick-Union, y explicar la estructura de bosque que mantiene la segunda.
4. **Analizar** por qué Quick-Union degenera a $O(n)$ y cómo lo corrigen la unión por tamaño y la compresión de caminos.
5. **Resolver** un problema de componentes conexas eligiendo la variante adecuada según el costo.

## Verificación de Dependencias


In [ ]:
import random
import sys
import time
import unittest

print("🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")


## 1. ¿Qué es un Tipo de Dato Abstracto (TDA)?

Un **Tipo de Dato Abstracto (TDA)** —o *Abstract Data Type (ADT)* en inglés— es una forma de pensar sobre los datos **separando el *qué* del *cómo***:

| Dimensión | Pregunta | Responsable |
|-----------|----------|-------------|
| **Interfaz (API)** | ¿Qué operaciones puedo hacer? | El diseñador / usuario |
| **Implementación** | ¿Cómo se ejecutan esas operaciones? | El programador |

### Analogía: Un cajero automático 🏧

Tú sabes que puedes *depositar*, *retirar* y *consultar saldo*, pero **no sabes** (ni necesitas saber) cómo el banco guarda esa información internamente. Eso es abstracción.

### ¿Por qué importa en esta unidad?

Cada estructura de datos que estudiaremos tiene:
1. Una **API clara** (contrato con el usuario)
2. **Una o más implementaciones** con distintos costos
3. **Casos de uso** donde una implementación es mejor que otra

```
     ┌─────────────────────────────────────┐
     │           INTERFAZ (API)            │
     │   union(p, q)   connected(p, q)     │
     └──────────────┬──────────────────────┘
                    │ oculta los detalles
          ┌─────────┴──────────┐
          │                    │
   Quick Find            Quick Union
  (implementación 1)  (implementación 2)
```

---

## 2. El Problema: Conectividad Dinámica

Imagina que tienes **N objetos** (nodos, computadores, personas, píxeles…).

Necesitas responder eficientemente dos tipos de preguntas:

- **`union(p, q)`** → Conectar el objeto `p` con el objeto `q`
- **`connected(p, q)`** → ¿Existe algún camino que conecte `p` con `q`?

### Ejemplos del mundo real

| Dominio | Objetos | Conexión |
|---------|---------|----------|
| Redes sociales | Personas | Amistad mutua: ¿hay una cadena de amigos entre tú y otra persona? |
| Juegos en línea | Jugadores | Estar en el mismo clan o la misma partida |
| Editor de imágenes | Píxeles | Ser vecinos del mismo color: el «bote de pintura» rellena un componente |
| Mapa de carreteras | Ciudades | Camino directo: ¿se puede ir de Talca a Curicó sin salir de la red? |
| Redes de computadores | Equipos | Cable o WiFi |

### Propiedades de "estar conectado"

La relación de conectividad es una **relación de equivalencia**, lo que implica:

- **Reflexiva:** `p` está conectado consigo mismo
- **Simétrica:** Si `p` está conectado con `q`, entonces `q` está conectado con `p`
- **Transitiva:** Si `p–q` y `q–r`, entonces `p–r`

Esto nos permite agrupar objetos en **componentes conectados** (conjuntos disjuntos).

### No toda relación sirve

Union-Find solo modela relaciones que cumplen las **tres** propiedades. Basta que falle una
para que la idea de «grupos» deje de tener sentido:

| Relación | ¿Qué falla? | Ejemplo concreto |
|----------|-------------|------------------|
| «es hermano/a de» | **No es reflexiva** | Ana es hermana de Luis, pero Ana **no** es hermana de Ana: nadie es su propio hermano |
| «sigue a» (en Instagram) | No es simétrica | Tú sigues a un futbolista famoso; él no te sigue a ti |
| «mide a lo más 5 cm de diferencia con» | No es transitiva | Ana mide 160, Beto 164 y Carla 168: Ana–Beto y Beto–Carla cumplen, pero Ana y Carla difieren en 8 cm |

Con «es hermano/a de», la consulta `connected(ana, ana)` debería responder **falso**, pero en
Union-Find todo objeto está en su propio componente desde el inicio, así que respondería
verdadero. La estructura responde otra pregunta: «¿están en el mismo grupo?» (por ejemplo,
«¿son de la misma familia?»), que sí es reflexiva.

> 🎙️ **[PAUSA PROFESOR]** *"«Ser compañeros de curso» ¿es una relación de equivalencia? ¿Y «haber jugado juntos alguna partida»?"*
> *La primera sí, si se acepta que cada uno es compañero de sí mismo: reflexiva, simétrica y transitiva, y los grupos son los cursos. La segunda no es transitiva: A jugó con B y B con C, pero A y C pueden no haber jugado nunca. Union-Find sí puede responder «¿hay una cadena de partidas que los une?», que es su cierre transitivo.*

### Visualización inicial

```
Objetos:  0  1  2  3  4  5  6  7  8  9

Antes de cualquier unión → cada objeto es su propio componente:
{0} {1} {2} {3} {4} {5} {6} {7} {8} {9}

Después de union(4,3), union(3,8), union(6,5):
{0} {1} {2} {3,4,8} {5,6} {7} {9}
```

---

## 3. API de Union-Find

Definimos el **contrato** de nuestra estructura antes de escribir una sola línea de implementación.

```
╔══════════════════════════════════════════════════════╗
║              API  —  Union Find                      ║
╠══════════════════════════════════════════════════════╣
║  UF(n)              Inicializar con n objetos (0..n-1)║
║  union(p, q)        Agregar conexión entre p y q     ║
║  connected(p, q)    ¿Están p y q en el mismo         ║
║                     componente conectado?             ║
║  find(p)            Retorna el id del componente      ║
║                     al que pertenece p               ║
║  count()            Número de componentes            ║
╚══════════════════════════════════════════════════════╝
```

> **Nota:** La API es la misma para Quick Find y Quick Union. Lo que cambia es la implementación interna.

In [ ]:
# 📋 Clase base abstracta — define la interfaz (el contrato)
# Todos los Union-Find deben implementar estos métodos

from abc import ABC, abstractmethod

class UnionFind(ABC):
    """
    Tipo de Dato Abstracto: Union-Find (Conjuntos Disjuntos)
    Resuelve el problema de conectividad dinámica.
    """

    @abstractmethod
    def union(self, p: int, q: int) -> None:
        """Conecta el componente que contiene p con el que contiene q."""
        pass

    @abstractmethod
    def find(self, p: int) -> int:
        """Retorna el identificador del componente al que pertenece p."""
        pass

    def connected(self, p: int, q: int) -> bool:
        """Retorna True si p y q están en el mismo componente."""
        return self.find(p) == self.find(q)   # ← reutiliza find()

    @abstractmethod
    def count(self) -> int:
        """Retorna el número de componentes conectados."""
        pass

print("✅ Clase abstracta UnionFind definida correctamente")

> 💡 **Punto clave:** `connected()` está implementado en la clase base usando `find()`. Esto significa que cualquier subclase que implemente `find()` **hereda** `connected()` gratis. Esto es el poder de la abstracción.

---

## 4. Quick Find

### Idea principal

Mantener un arreglo `id[]` donde **`id[p]` es el identificador del componente de `p`**.

- Dos objetos `p` y `q` están conectados **si y solo si** `id[p] == id[q]`
- `find(p)` es trivial: retorna `id[p]` → **O(1)**
- `union(p, q)` debe actualizar todos los que tenían el id de `p` → **O(N)**

### Ejemplo paso a paso

```
Inicial (N=10):
índice:  0  1  2  3  4  5  6  7  8  9
id[]:    0  1  2  3  4  5  6  7  8  9

union(4, 3):  todos los id[i]==4 pasan a ser 3
id[]:    0  1  2  3  3  5  6  7  8  9
                  ↑  ↑
union(3, 8):  todos los id[i]==3 pasan a ser 8
id[]:    0  1  2  8  8  5  6  7  8  9
                  ↑  ↑        ↑
union(6, 5):  todos los id[i]==6 pasan a ser 5
id[]:    0  1  2  8  8  5  5  7  8  9
                           ↑  ↑
connected(4, 8)?  id[4]=8, id[8]=8  → True ✅
connected(4, 6)?  id[4]=8, id[6]=5  → False ❌
```

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# IMPLEMENTACIÓN: Quick Find
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class QuickFind(UnionFind):
    """
    Quick Find: find es O(1), union es O(N).
    
    Estructura interna:
        id[i] = identificador del componente al que pertenece i
    Invariante:
        p y q están conectados  <=>  id[p] == id[q]
    """

    def __init__(self, n: int):
        # Cada objeto empieza siendo su propio componente
        self._id = list(range(n))   # id[i] = i
        self._count = n             # n componentes al inicio

    def find(self, p: int) -> int:
        """O(1) — acceso directo al arreglo."""
        return self._id[p]

    def union(self, p: int, q: int) -> None:
        """
        O(N) — debe recorrer todo el arreglo.
        Cambia todos los id[] que coincidan con pid al valor qid.
        """
        pid = self.find(p)   # componente de p
        qid = self.find(q)   # componente de q

        if pid == qid:       # ya están conectados → no hacer nada
            return

        # Recorrer todo el arreglo y reemplazar pid → qid
        for i in range(len(self._id)):
            if self._id[i] == pid:
                self._id[i] = qid

        self._count -= 1     # se redujo un componente

    def count(self) -> int:
        return self._count

    def __repr__(self):
        return f"id[] = {self._id}  |  componentes = {self._count}"


print("✅ Clase QuickFind definida")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DEMOSTRACIÓN: Quick Find en acción
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 55)
print("  DEMO Quick Find — N = 10 objetos")
print("=" * 55)

qf = QuickFind(10)
print(f"\nInicial:    {qf}")

operaciones = [
    ("union", 4, 3),
    ("union", 3, 8),
    ("union", 6, 5),
    ("union", 9, 4),
    ("union", 2, 1),
]

for op, p, q in operaciones:
    qf.union(p, q)
    print(f"union({p},{q}) → {qf}")

print()
consultas = [(0, 7), (8, 9), (5, 4), (3, 4)]
for p, q in consultas:
    resultado = qf.connected(p, q)
    simbolo = "✅" if resultado else "❌"
    print(f"connected({p},{q}) = {resultado}  {simbolo}")

### Visualización: estado del arreglo `id[]`

In [ ]:
# Visualización en texto del arreglo id[] de Quick Find

def mostrar_id(id_arr, titulo="", anterior=None):
    """
    Imprime id[] alineado bajo sus índices y agrupa los objetos por componente.

    Parámetros:
        id_arr (list[int]): arreglo id[] de Quick Find.
        titulo (str): encabezado opcional.
        anterior (list[int] | None): estado previo; si se entrega, marca con ↑
            las posiciones que cambiaron.
    """
    ancho = len(str(max(len(id_arr) - 1, *id_arr)))

    def fila(valores):
        return "  ".join(f"{v:>{ancho}}" for v in valores)

    componentes = {}
    for i, comp in enumerate(id_arr):
        componentes.setdefault(comp, []).append(i)

    if titulo:
        print(titulo)
    print(f"  índice: {fila(range(len(id_arr)))}")
    print(f"  id[]:   {fila(id_arr)}")
    if anterior is not None:
        print(f"          {fila('↑' if a != b else ' ' for a, b in zip(anterior, id_arr))}")
    print("  " + " ".join("{" + ",".join(map(str, g)) + "}" for g in componentes.values()))


# Estado final del Quick Find de la demo
mostrar_id(qf._id, titulo="Quick Find — Estado final de id[]")

In [ ]:
# Evolución paso a paso del arreglo id[] — ↑ marca lo que cambió cada union()

qf2 = QuickFind(10)
mostrar_id(qf2._id, titulo="Inicial")

pasos = [(4,3), (3,8), (6,5), (9,4), (2,1)]
for p, q in pasos:
    antes = list(qf2._id)
    qf2.union(p, q)
    print()
    mostrar_id(qf2._id, titulo=f"union({p},{q})", anterior=antes)

### ⚠️ El problema de Quick Find

**`union()` es demasiado lento.** Si realizamos N operaciones `union` sobre N objetos:

```
Costo total ≈ N × N = N²  accesos al arreglo
```

Para N = 10⁹ objetos (escala moderna) → **10¹⁸ operaciones** → inviable.

| Operación | Quick Find |
|-----------|------------|
| `find(p)` | O(1) |
| `union(p, q)` | **O(N)** ← cuello de botella |
| `connected(p, q)` | O(1) |

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: *"Quick-Find responde `find` en O(1). ¿Por qué no lo usamos siempre?"* — Porque su `union` es O(n): con muchas uniones, que es el caso real, el total se vuelve cuadrático.

---

## 5. Quick Union

### Idea principal: árboles perezosos

Usamos el mismo arreglo `id[]`, pero ahora lo interpretamos como un **bosque de árboles**:

- `id[i]` = **padre de `i`** en el árbol
- Si `id[i] == i` → `i` es la **raíz** de su árbol (su propio padre)
- Cada árbol representa un componente conectado
- La **raíz** del árbol es el identificador del componente

```
Arreglo id[]:
 índice:  0  1  2  3  4  5  6  7  8  9
 id[]:    0  1  9  4  9  6  6  7  8  9

Árbol resultante:
          9        6        0  1  7  8
         / \       |
        2   4      5
            |
            3
```

### Operaciones

- **`find(p)`:** Seguir los punteros padre hasta llegar a la raíz → **O(profundidad)**
- **`union(p, q)`:** Encontrar raíces de p y q, hacer que una apunte a la otra → **O(profundidad)**

```
union(p, q):   find raíz de p → rp
               find raíz de q → rq
               id[rp] = rq         ← solo UN cambio
```

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# IMPLEMENTACIÓN: Quick Union
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class QuickUnion(UnionFind):
    """
    Quick Union: árbol con estructura de bosque.
    
    Estructura interna:
        id[i] = padre de i en el árbol
        id[i] == i  <=>  i es raíz
    
    Costo: O(profundidad) para find y union.
    Problema: el árbol puede degenerar en lista enlazada.
    """

    def __init__(self, n: int):
        # Al inicio, cada nodo apunta a sí mismo (es su propia raíz)
        self._id = list(range(n))   # id[i] = i
        self._count = n

    def _root(self, p: int) -> int:
        """
        Seguir los punteros padre hasta llegar a la raíz.
        La raíz se reconoce porque id[raíz] == raíz.
        O(profundidad del árbol)
        """
        while self._id[p] != p:      # mientras p no sea raíz
            p = self._id[p]          # subir un nivel hacia la raíz
        return p

    def find(self, p: int) -> int:
        """La identidad del componente es la raíz del árbol."""
        return self._root(p)

    def union(self, p: int, q: int) -> None:
        """
        O(profundidad) — solo cambia UN elemento del arreglo.
        Hace que la raíz de p apunte a la raíz de q.
        """
        rp = self._root(p)   # raíz del árbol de p
        rq = self._root(q)   # raíz del árbol de q

        if rp == rq:         # ya están en el mismo árbol
            return

        self._id[rp] = rq    # ← ¡solo UN cambio!
        self._count -= 1

    def count(self) -> int:
        return self._count

    def __repr__(self):
        return f"id[] = {self._id}  |  componentes = {self._count}"


print("✅ Clase QuickUnion definida")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DEMOSTRACIÓN: Quick Union en acción
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 55)
print("  DEMO Quick Union — N = 10 objetos")
print("=" * 55)

qu = QuickUnion(10)
print(f"\nInicial:    {qu}")

for op, p, q in operaciones:   # mismas operaciones que antes
    qu.union(p, q)
    print(f"union({p},{q}) → {qu}")

print()
for p, q in consultas:
    resultado = qu.connected(p, q)
    simbolo = "✅" if resultado else "❌"
    print(f"connected({p},{q}) = {resultado}  {simbolo}")

In [ ]:
# ── Visualización en texto del bosque de Quick Union ────────────────────────
# Dibujamos cada árbol como el comando `tree` de Linux muestra las carpetas:
# la raíz arriba y los hijos colgando con ├── y └──. La sangría de cada nodo
# es su profundidad, que es justo lo que determina el costo de find().

def mostrar_bosque(id_arr, titulo=""):
    """
    Imprime el bosque de Quick Union como el comando `tree` muestra carpetas.

    Cada raíz (id[r] == r) encabeza su árbol entre corchetes y los hijos cuelgan
    con ├── y └──. La sangría de un nodo es su profundidad: los pasos que da
    find() para llegar a la raíz.

    Parámetros:
        id_arr (list[int]): arreglo id[] de Quick Union.
        titulo (str): encabezado opcional.

    Complejidad:
        Temporal: O(n) — cada nodo se imprime una vez.
        Espacial: O(n) — listas de hijos y pila de recursión.
    """
    hijos = {i: [] for i in range(len(id_arr))}
    raices = []
    for i, padre in enumerate(id_arr):
        if padre == i:
            raices.append(i)
        else:
            hijos[padre].append(i)

    def altura(nodo):
        return max((1 + altura(h) for h in hijos[nodo]), default=0)

    def imprimir_hijos(nodo, sangria):
        for k, hijo in enumerate(hijos[nodo]):
            ultimo = k == len(hijos[nodo]) - 1
            print(sangria + ("└── " if ultimo else "├── ") + str(hijo))
            imprimir_hijos(hijo, sangria + ("    " if ultimo else "│   "))

    if titulo:
        print(titulo)
    for r in raices:
        print(f"[{r}]")
        imprimir_hijos(r, "")
    print(f"{len(raices)} árbol(es) · altura máxima {max(altura(r) for r in raices)}")


mostrar_bosque(qu._id, "Quick Union — árbol final")

### ⚠️ El problema de Quick Union

El árbol puede volverse **muy alto** (degenerar). En el peor caso, todos los nodos forman una cadena lineal:

```
union(0,1), union(0,2), union(0,3), union(0,4) ...

    4
    |
    3
    |
    2
    |
    1
    |
    0   ← raíz
```

En ese caso `find(4)` tarda **N pasos** → O(N), igual de lento que Quick Find.

| Operación | Quick Find | Quick Union |
|-----------|------------|-------------|
| `find(p)` | O(1) | O(N) peor caso |
| `union(p, q)` | O(N) | O(N) peor caso |
| `connected(p, q)` | O(1) | O(N) peor caso |

In [ ]:
# Demostración del peor caso de Quick Union

qu_peor = QuickUnion(6)
print("Secuencia que genera árbol degenerado:")
print(f"Inicial: {qu_peor}")

for i in range(5, 0, -1):          # 5→4, 4→3, 3→2, 2→1, 1→0
    qu_peor.union(i, i-1)
    print(f"union({i},{i-1}) → {qu_peor}")

print()
print("Árbol resultante:")
mostrar_bosque(qu_peor._id, "Quick Union — Peor caso (cadena lineal)")

---

## 6. Dos mejoras: unión por tamaño y compresión de caminos

El problema de Quick Union no es la idea del bosque, sino que los árboles pueden quedar muy
altos. Dos cambios pequeños lo corrigen.

### Mejora 1 — Unión por tamaño (*Weighted Quick Union*)

En `union`, en vez de colgar siempre la raíz de `p` bajo la de `q`, se cuelga **el árbol más
chico bajo el más grande**. Para eso basta guardar el tamaño de cada árbol en un segundo arreglo.

```
Quick Union:  cuelga el primero bajo el segundo      Con tamaño: el chico bajo el grande

     árbol grande                                          árbol grande
          │                                                 /        \
       árbol chico        ← la altura crece           (resto)     árbol chico   ← no crece
```

Un nodo solo baja un nivel cuando su árbol se une a otro **al menos igual de grande**, o sea
cuando el tamaño de su árbol al menos se duplica. Eso puede pasar a lo más log₂ N veces:

> 📌 **Garantía:** con unión por tamaño, la altura de cualquier árbol es a lo más **log₂ N**.
> `find` y `union` pasan de O(N) a **O(log N)** en el peor caso.

### Mejora 2 — Compresión de caminos

Cada vez que `find(p)` sube hasta la raíz, aprovecha el viaje para **acortar el camino**:
hace que cada nodo visitado apunte a su abuelo (o directo a la raíz). Es una sola línea extra
dentro del ciclo de `_root`:

```python
while id[p] != p:
    id[p] = id[id[p]]    # ← p ahora apunta a su abuelo: el camino se acorta a la mitad
    p = id[p]
```

Los árboles quedan casi planos. Con las dos mejoras juntas, M operaciones sobre N objetos
cuestan casi M: el costo por operación es **prácticamente constante** (crece como la inversa
de la función de Ackermann, α(N), que es menor que 5 para cualquier N imaginable).

> 🎙️ **[PAUSA PROFESOR]** *"Con unión por tamaño, ¿cuántos objetos necesito como mínimo para que exista un árbol de altura 3?"*
> *8. Para que la altura suba en uno hay que unir dos árboles de la misma altura; el de altura h necesita al menos 2^h nodos.*

La implementación de la unión por tamaño está en `unionfind.py` (`WeightedQuickUnion`), que
usarás como caja negra en el laboratorio. Implementarla tú es el **bonus** del laboratorio.
Aquí medimos cuánto se gana.


In [ ]:
# El peor caso de Quick Union, medido contra la unión por tamaño de la librería
from unionfind import WeightedQuickUnion

print(f"{'N':>6} │ {'Quick Union':>12} │ {'Weighted (librería)':>20} │ {'mejora':>8}")
print("─" * 56)
for n in (500, 1000, 2000, 4000):
    tiempos = []
    for clase in (QuickUnion, WeightedQuickUnion):
        uf = clase(n)
        t0 = time.perf_counter()
        for i in range(n - 1, 0, -1):      # la secuencia que degenera a Quick Union
            uf.union(i, i - 1)
        for _ in range(n):                 # N consultas sobre el nodo más profundo
            uf.find(n - 1)
        tiempos.append((time.perf_counter() - t0) * 1000)
    print(f"{n:>6} │ {tiempos[0]:>9.1f} ms │ {tiempos[1]:>17.1f} ms │ {tiempos[0] / tiempos[1]:>7.0f}×")

print("\n💡 Al duplicar N, Quick Union tarda unas 4 veces más: su costo total es cuadrático.")
print("💡 Con unión por tamaño solo se duplica: cada operación cuesta a lo más log₂ N pasos.")


---

## 7. Comparación y Análisis de Costo

### Tabla comparativa: accesos al arreglo `id[]`

| Algoritmo | Inicializar | `union` | `find` |
|-----------|-------------|---------|--------|
| **Quick Find** | N | N | 1 |
| **Quick Union** | N | N† | N† |
| **Weighted Quick Union** | N | log N† | log N |
| **Weighted + compresión de caminos** | N | casi 1 (amortizado) | casi 1 (amortizado) |

> † incluye el costo de `find`. En Quick Union, en el peor caso el árbol tiene altura N.

### ¿Cuándo conviene cada uno?

| Situación | Mejor opción |
|-----------|-------------|
| Muchas consultas `connected`, pocas uniones | **Quick Find** |
| Pocos elementos, árbol balanceado esperado | **Quick Union** |
| Producción / gran escala | **Weighted Quick Union + compresión de caminos** (Sección 6) |

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Benchmark: comparación empírica de tiempos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def benchmark(clase, n, operaciones_union, operaciones_find, semilla=42):
    """Mide el tiempo de N operaciones union y find."""
    random.seed(semilla)
    uf = clase(n)

    pares_union = [(random.randint(0, n-1), random.randint(0, n-1))
                   for _ in range(operaciones_union)]
    pares_find  = [(random.randint(0, n-1), random.randint(0, n-1))
                   for _ in range(operaciones_find)]

    inicio = time.perf_counter()
    for p, q in pares_union:
        uf.union(p, q)
    for p, q in pares_find:
        uf.connected(p, q)
    fin = time.perf_counter()

    return fin - inicio

def mostrar_barras(filas, unidad="ms", ancho=40):
    """
    Imprime un gráfico de barras horizontal: la barra del valor máximo mide `ancho`.

    Parámetros:
        filas (list[tuple[str, float]]): pares (etiqueta, valor).
        unidad (str): unidad que se muestra junto a cada valor.
        ancho (int): caracteres de la barra más larga.
    """
    maximo = max(valor for _, valor in filas) or 1
    margen = max(len(etiqueta) for etiqueta, _ in filas)
    for etiqueta, valor in filas:
        barra = "█" * round(valor / maximo * ancho) or ("▏" if valor > 0 else "")
        print(f"{etiqueta:>{margen}} │{barra:<{ancho}} {valor:8.2f} {unidad}")


tamanios = [500, 1000, 2000, 4000, 8000]
filas = []

for n in tamanios:
    t_qf = benchmark(QuickFind,  n, n, n*2) * 1000
    t_qu = benchmark(QuickUnion, n, n, n*2) * 1000
    print(f"N={n:5d}  |  QuickFind: {t_qf:6.2f} ms  |  QuickUnion: {t_qu:6.2f} ms")
    filas += [(f"N={n} QF", t_qf), ("QU", t_qu)]

# Gráfico comparativo en texto
print("\nQuick Find vs Quick Union — Tiempo vs N\n")
mostrar_barras(filas)

---

## 8. Ejercicios 💪

## 🧪 Ejercicio 1 ⭐: Trazar a mano

Dado un arreglo inicial con `N = 8` objetos, aplica las siguientes operaciones de **Quick Find** y dibuja el arreglo `id[]` después de cada paso:

```
union(0, 2)
union(1, 3)
union(0, 3)
union(4, 7)
union(3, 5)
```

¿Cuántos componentes quedan? ¿Es `connected(2, 5)` verdadero?

---

## 🧪 Ejercicio 2 ⭐: Aplicar una secuencia de uniones

Completa la función `union_sequence` que recibe una lista de pares y devuelve:
- El número de componentes final
- El contenido del arreglo `id[]`

In [ ]:
# EJERCICIO 2 — completa las líneas marcadas con TODO

def union_sequence(n: int, pares: list, algoritmo) -> dict:
    """
    Aplica una secuencia de uniones y retorna el estado final.
    
    Parámetros:
        n        : número de objetos
        pares    : lista de tuplas (p, q) a unir
        algoritmo: QuickFind o QuickUnion
    """
    uf = algoritmo(n)

    for p, q in pares:
        # TODO: aplica la unión
        pass

    return {
        "componentes": None,    # TODO: obtén el número de componentes
        "id": None,             # TODO: retorna el arreglo id[]
    }


# Prueba tu solución:
pares_prueba = [(0,2), (1,3), (0,3), (4,7), (3,5)]

resultado_qf = union_sequence(8, pares_prueba, QuickFind)
resultado_qu = union_sequence(8, pares_prueba, QuickUnion)

print("Quick Find  →", resultado_qf)
print("Quick Union →", resultado_qu)

In [ ]:
# Pruebas automáticas del ejercicio 2 (unittest)

class TestEjercicio2(unittest.TestCase):
    """union_sequence aplica las uniones y entrega el estado final."""

    PARES = [(0, 2), (1, 3), (0, 3), (4, 7), (3, 5)]

    def grupos(self, id_arr, algoritmo):
        """Agrupa los objetos por componente, sin depender de qué identificador se usó."""
        uf = algoritmo(len(id_arr))
        uf._id = list(id_arr)
        return sorted(sorted(i for i in range(len(id_arr)) if uf.find(i) == uf.find(r))
                      for r in set(uf.find(i) for i in range(len(id_arr))))

    def test_1_componentes(self):
        """con las 5 uniones del enunciado quedan 3 componentes"""
        for algoritmo in (QuickFind, QuickUnion):
            with self.subTest(algoritmo.__name__):
                self.assertEqual(union_sequence(8, self.PARES, algoritmo)["componentes"], 3)

    def test_2_agrupacion(self):
        """los grupos son {0,1,2,3,5}, {4,7} y {6}"""
        for algoritmo in (QuickFind, QuickUnion):
            with self.subTest(algoritmo.__name__):
                id_final = union_sequence(8, self.PARES, algoritmo)["id"]
                self.assertEqual(self.grupos(id_final, algoritmo), [[0, 1, 2, 3, 5], [4, 7], [6]])

    def test_3_sin_uniones(self):
        """sin pares, cada objeto es su propio componente"""
        self.assertEqual(union_sequence(4, [], QuickFind)["componentes"], 4)


resultado = unittest.main(argv=["ignorado", "TestEjercicio2"], exit=False, verbosity=2)
if resultado.result.wasSuccessful():
    print("\n🎉 Todos los casos pasaron")


In [ ]:
# SOLUCIÓN EJERCICIO 2 — ejecuta solo después de intentarlo

def union_sequence_sol(n, pares, algoritmo):
    uf = algoritmo(n)
    for p, q in pares:
        uf.union(p, q)
    return {"componentes": uf.count(), "id": uf._id}

r_qf = union_sequence_sol(8, pares_prueba, QuickFind)
r_qu = union_sequence_sol(8, pares_prueba, QuickUnion)

print("✅ Quick Find  →", r_qf)
print("✅ Quick Union →", r_qu)

# Verificación
qf_test = QuickFind(8)
for p, q in pares_prueba:
    qf_test.union(p, q)
print(f"\nconnected(2, 5) con QuickFind = {qf_test.connected(2, 5)}")

### Ejercicio 3 — Análisis crítico

Considera la siguiente afirmación:

> *"Quick Union siempre es mejor que Quick Find porque `union()` solo modifica un elemento del arreglo."*

**¿Es correcta esta afirmación?** Justifica tu respuesta con un contraejemplo o con análisis de casos.

---

### Ejercicio 4 — Para pensar 🧠 (Challenge)

¿Cómo modificarías `QuickUnion` para que el árbol no crezca demasiado?
Pista: piensa en qué árbol conviene hacer raíz del otro durante `union()`.

*(Es la **unión por tamaño** de la Sección 6. Implementarla es el bonus del laboratorio.)*

---

## 📌 Resumen de la clase

| Concepto | Definición corta |
|----------|------------------|
| **TDA** | Separación entre *qué* hace una estructura y *cómo* lo hace |
| **Conectividad dinámica** | Problema de mantener componentes conectados con `union` y `connected` |
| **Quick Find** | `id[p]` = componente. `find` O(1), `union` O(N) |
| **Quick Union** | `id[p]` = padre. Árbol implícito. Ambas O(profundidad), peor caso O(N) |
| **Unión por tamaño** | Colgar el árbol chico bajo el grande: altura ≤ log₂ N, operaciones O(log N) |
| **Compresión de caminos** | Acortar el camino en cada `find`: costo casi constante, O(α(N)) amortizado |

---

### 📚 Bibliografía

- Sedgewick, R. & Wayne, K. *Algorithms, 4th Edition*. Princeton. (Capítulo 1.5)
- Coursera: [Algorithms Part I — Union Find](https://www.coursera.org/learn/algorithms-part1)

---
*Notebook preparado para la Unidad 2 — Estructuras de Datos*

## 📚 Lecturas Recomendadas y Práctica

### Textos del curso

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 19 | Estructuras de datos para conjuntos disjuntos |
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 14.7 | Union-Find y sus aplicaciones |
| Miller & Ranum (M&R) — *Problem Solving with Algorithms and Data Structures Using Python* | 2011 | Cap. 3 | Tipos de datos abstractos |

### Recursos gratuitos en línea

- 🌐 [VisuAlgo — Union-Find (DSU)](https://visualgo.net/en/ufds) — union por tamaño y compresión de caminos, paso a paso.
- 🎬 [Algorithms, Part I — Sedgewick (Princeton)](https://www.coursera.org/learn/algorithms-part1) — el curso de referencia de este ramo.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe la etiqueta en **Tags** y ajusta **Rating**.

**Escala de dificultad orientativa para este curso:**

| Rating | Nivel | Descripción |
|--------|-------|-------------|
| 800 | ⭐ | Aplicación directa — la mayoría puede resolverlo |
| 1000–1200 | ⭐⭐ | Requiere una pequeña adaptación |
| 1300+ | ⭐⭐⭐ | Combina la idea con otra — desafío |

**Problemas recomendados para este tópico:**

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [1249B1 — Books Exchange (easy version)](https://codeforces.com/problemset/problem/1249/B1) | ⭐⭐ 1000 | Componentes conexas en una permutación: el caso más limpio de DSU |
| 2 | [217A — Ice Skating](https://codeforces.com/problemset/problem/217/A) | ⭐⭐ 1200 | Contar componentes conexas: la aplicación canónica |
| 3 | [277A — Learning Languages](https://codeforces.com/problemset/problem/277/A) | ⭐⭐⭐ 1400 | DSU con un truco de modelado: las personas y los idiomas en el mismo bosque |

⚠️ Los dos primeros son el **mínimo esperado**. Los demás son desafío opcional.